In [1]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
csv_candidates = ['performance_history.csv', 'orca_executor/performance_history.csv']
csv_path = next((p for p in csv_candidates if os.path.exists(p)), csv_candidates[0])
df = pd.read_csv(csv_path)

df['entry_time'] = pd.to_datetime(df['entry_time'])
df.set_index('entry_time', inplace=True)

# Ensure non-negative accrued fees (guarding legacy uninitialized-tick records)
df['earned_fees_usd'] = df['earned_fees_usd'].clip(lower=0.0)

# Base ML Strategy Profit (Isolated $1,000 deployment per hour)
df['net_profit_usd'] = df['inventory_pnl_usd'] + df['earned_fees_usd']
df['cumulative_profit_isolated'] = df['net_profit_usd'].cumsum()
df['cumulative_fees'] = df['earned_fees_usd'].cumsum()

# ==========================================
# 2. COMPOUND PORTFOLIO & TRANSACTION COSTS
# ==========================================
SWAP_FEE_PCT = 0.0005  # 0.05% rebalancing swap cost & slippage
GAS_FEE_USD = 0.01     # ~$0.01 Solana priority + base transaction fees

df['hourly_return_pct'] = df['net_profit_usd'] / 1000.0
df['real_hourly_return_pct'] = df['hourly_return_pct'] - SWAP_FEE_PCT - (GAS_FEE_USD / 1000.0)

df['compound_balance'] = 1000.0 * (1.0 + df['real_hourly_return_pct']).cumprod()
df['cumulative_profit_compound'] = df['compound_balance'] - 1000.0

# Benchmark 1: 100% SOL Buy & Hold
initial_sol_price = df['entry_price'].iloc[0]
df['hold_100_sol_value'] = (1000.0 / initial_sol_price) * df['exit_price']
df['cumulative_profit_hold_100'] = df['hold_100_sol_value'] - 1000.0

# Benchmark 2: 50% SOL / 50% USDC Static Portfolio
df['hold_5050_value'] = 500.0 + (500.0 / initial_sol_price) * df['exit_price']
df['cumulative_profit_hold_5050'] = df['hold_5050_value'] - 1000.0

df['breakeven'] = 0.0

# ==========================================
# 3. QUANTITATIVE PERFORMANCE METRICS
# ==========================================
win_rate = (df['net_profit_usd'] > 0).mean() * 100.0
in_range_rate = (df['status'] == 'IN_RANGE').mean() * 100.0
total_hours = len(df)

mean_return = df['real_hourly_return_pct'].mean()
std_return = df['real_hourly_return_pct'].std()
sharpe_ratio = (mean_return / std_return) * np.sqrt(24 * 365) if std_return > 0 else 0.0

downside_returns = df['real_hourly_return_pct'].clip(upper=0.0)
downside_std = np.sqrt((downside_returns ** 2).mean())
sortino_ratio = (mean_return / downside_std) * np.sqrt(24 * 365) if downside_std > 0 else 0.0

rolling_max = df['compound_balance'].cummax()
drawdown = (df['compound_balance'] - rolling_max) / rolling_max
max_drawdown = drawdown.min() * 100.0

print(f"STRATEGY PERFORMANCE REPORT ({total_hours} Hours):")
print(f"Win Rate (Gross):       {win_rate:.1f}%")
print(f"In-Range Hit Rate:      {in_range_rate:.1f}%")
print(f"Sharpe Ratio (Ann.):    {sharpe_ratio:.2f}")
print(f"Sortino Ratio (Ann.):   {sortino_ratio:.2f}")
print(f"Max Drawdown (Net):     {max_drawdown:.2f}%")
print("-" * 40)

# ==========================================
# 4. INTERACTIVE PLOTLY DASHBOARD
# ==========================================
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=("Cumulative PnL Comparison: AI Bot vs. Benchmarks", "Cumulative CLMM Fee Cashflow (USD)"),
    row_heights=[0.7, 0.3]
)

fig.add_trace(go.Scatter(
    x=df.index, y=df['breakeven'],
    mode='lines', name='Initial $1,000 Breakeven',
    line=dict(color='white', width=1.5, dash='dashdot'),
    opacity=0.6
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['cumulative_profit_isolated'],
    mode='lines', name='AI Bot (Isolated $1,000/hr Gross)',
    line=dict(color='#38bdf8', dash='dot')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['cumulative_profit_compound'],
    mode='lines', name='AI Bot (Compounded Net of Fees & Slippage)',
    line=dict(color='#22c55e', width=2.5)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['cumulative_profit_hold_100'],
    mode='lines', name='Buy & Hold (100% SOL)',
    line=dict(color='#f97316', width=2, dash='dash')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['cumulative_profit_hold_5050'],
    mode='lines', name='Buy & Hold (50% SOL / 50% USDC)',
    line=dict(color='#eab308', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['cumulative_fees'],
    mode='lines', fill='tozeroy', name='Accrued CLMM Fees (USD)',
    line=dict(color='#a855f7')
), row=2, col=1)

fig.update_layout(
    height=800,
    width=1300,
    title_text="Orca Whirlpool CLMM Transformer Strategy — Performance & Fee Analytics",
    template="plotly_dark",
    hovermode="x unified"
)

fig.update_yaxes(tickformat="$.2f", ticks="outside", ticklen=10)
fig.update_xaxes(row=2, col=1, ticks="outside", ticklen=10)
fig.show()


STRATEGY PERFORMANCE REPORT (1500 Hours):
Win Rate (Gross):       73.9%
In-Range Hit Rate:      91.7%
Sharpe Ratio (Ann.):    44.12
Sortino Ratio (Ann.):   468.01
Max Drawdown (Net):     -4.88%
----------------------------------------
